# Local Green, Global Red III: 03 When Paths Compose
## What must hold before local path calculations compose?

**Author:** Jeremy H. Carroll  
**Date:** 2026-10-15  
**Version:** 1.5.2  

---

### 1. Question and Prediction

> **Question:** What sufficient conditions ensure that path kernels compose to yield the composite kernel?

**Prediction:** Does numerical agreement between composed and convolved kernels prove that the underlying factorization hypotheses hold?

The answer is **no**: hypotheses G1–G4 are independent structural requirements. Numerical agreement can occur coincidentally through internal cancellation even when structural hypotheses fail.

In [ ]:
import epk_gtla_overlay_core as core

print('Loaded production verification core:', core.__file__)

### 2. Graph Topology and Compatible Pairs

Consider the graph $a \to \{b, d\} \to c$:

```
      b
    /   \
  a       c
    \   /
      d
```

Compatible pairs are segments sharing an intermediate boundary vertex:
- Boundary $b$: pair $(ab, bc)$
- Boundary $d$: pair $(ad, dc)$

Hypothesis **G1** asserts a **bijection** between compatible path pairs and composite paths.

### 3. G1 Multiplicity Check: Pair-Set Equality vs. Production Bijection Audit

A naive implementation might check whether the set of compatible pairs equals the set of composite endpoints. However, set equality **discards multiplicity**! If two distinct composite IDs refer to the same pair, set equality passes, but the mapping is not a bijection.

In [ ]:
first_stage = (core.PathSegment('ab', 'a', 'b', 1, 1),)
second_stage = (core.PathSegment('bc', 'b', 'c', 2, 1),)

# Defective composite set: TWO distinct composite paths cover the SAME compatible pair!
duplicate_composites = (
    core.CompositePath('abc-1', 'ab', 'bc', 3, 1),
    core.CompositePath('abc-2', 'ab', 'bc', 3, 1),
)

# Set comparison
compatible_pairs = [('b', 'bc', 'ab')]
covered_pairs = [('b', 'bc', 'ab'), ('b', 'bc', 'ab')]
set_passes = set(compatible_pairs) == set(covered_pairs)

# Production core audit (enforces exact bijection via Counter)
report = core.boundary_bijection_report(first_stage, second_stage, duplicate_composites)

print(f'Naive Set Equality Check:      {set_passes} (MISLEADING PASS)')
print(f'Production Core is_bijection:  {report.is_bijection} (CORRECTLY REJECTED)')
print(f'Production Detected Issues:    {report.issues}')

### 4. G3 Failure with Zero Residual: Numerical Agreement Does Not Prove Hypotheses

Now consider an evaluation where Hypothesis G3 (verdict factorization) fails, yet the total composed kernel residual is exactly **0.0** due to internal cancelling discrepancies.

In [ ]:
first = (
    core.PathSegment('ab', 'a', 'b', 0, 1),
    core.PathSegment('ad', 'a', 'd', 0, 0),
)
second = (
    core.PathSegment('bc', 'b', 'c', 0, 1),
    core.PathSegment('dc', 'd', 'c', 0, 1),
)
joined = (
    core.CompositePath('abc', 'ab', 'bc', 0, 0),
    core.CompositePath('adc', 'ad', 'dc', 0, 1),
)

# Audit kernel composition with decay=0
audit = core.audit_kernel_composition(first, second, joined, decay=0)

print('Kernel Composition Audit:')
print(f'  G1 (Boundary Bijection):    {audit.g1_bijection}')
print(f'  G2 (Additive Cost):         {audit.g2_cost_additive}')
print(f'  G3 (Verdict Factorization): {audit.g3_evaluator_factorized}  <-- HYPOTHESIS FAILS!')
print(f'  Hypotheses Hold:            {audit.hypotheses_hold}')
print(f'  Composed Kernel Value:      {audit.composed_value}')
print(f'  Convolution Value:          {audit.convolution_value}')
print(f'  Residual:                   {audit.residual}  <-- EXACT ZERO!')
print(f'  Identity Holds:             {audit.identity_holds}')
print(f'  Reported Issues:            {audit.issues}')

### 5. Exercise, Worked Answer, and Evidence Boundary

**Exercise:**  
Explain why `audit.identity_holds` being true does not certify that the theorem applies.

---

**Worked Answer:**  
The composition identity theorem states: *if* G1, G2, G3 (and G4 if applicable) hold, *then* the composed kernel equals the convolution. When G3 fails, the theorem's premises are unsatisfied. That the residual happens to be 0.0 is an accidental numerical coincidence resulting from cancelling errors, not an instance of the theorem's guarantee.

---

**Evidence Boundary:**  
This lesson proves that boundary bijections must preserve multiplicity, and that numerical kernel residuals must not be conflated with hypothesis verification. It applies to finite graph path categories.